# 매도 재원 리밸런싱 검증
2026-10-20 고정 입력. 현재 정수 수량의 자금 검증과 저장된 과거 비교 결과를 점검하는 companion이다.
이는 과거 시점에 생성한 투자 신호의 백테스트가 아니다. 세 매수 후보의 원분석 보류는 해제하지 않는다.

## 자료와 방법
같은 폴더의 plan.json, historical_analogues.csv, stress_results.csv, evidence.json을 사용한다.
원시 가격 재계산은 저장된 SHA256과 일치하는 CSV를 준비한 뒤 저장소 tools/review_funded_rebalance.py를 실행한다.
가격 CSV는 계좌 원문과 함께 로컬 연구 폴더에 보관하며 여기에 복제하지 않는다.
현재 비중의 과거 투영에는 생존·선택 편향이 있고 조정종가는 당시 시점의 수정 이력을 보장하지 않는다.


In [1]:
import json
from pathlib import Path
import pandas as pd

root = Path.cwd()
folder = root if (root / "plan.json").exists() else root / "Docs/evidence/funded-rebalance-20261020"
plans = json.loads((folder / "plan.json").read_text(encoding="utf8"))
historical = pd.read_csv(folder / "historical_analogues.csv")
stress = pd.read_csv(folder / "stress_results.csv")
evidence = json.loads((folder / "evidence.json").read_text(encoding="utf8"))
assert evidence["analysis_as_of"] == "2026-10-20"
assert len(historical) == 18 and len(stress) == 36
print("입력 기준일:", evidence["analysis_as_of"], "| 과거 비교:", len(historical), "| 스트레스:", len(stress))


입력 기준일: 2026-10-20 | 과거 비교: 18 | 스트레스: 36


## 자금과 정수 수량
예상 매도대금은 현재 주문가능금액으로 주입하지 않는다. US 22%는 실제 세금 추정이 아닌 매도액 전체 충당 상한이다.

In [2]:
for market, plan in plans.items():
    ledger = plan["cash_ledger"]
    expected = ledger["initial_cash_krw"] + ledger["net_sale_funding_krw"] - ledger["buy_total_krw"]
    assert abs(expected - ledger["after_buys_cash_krw"]) < 0.01
    assert expected >= ledger["reserve_krw"] + ledger["extra_cash_margin_krw"]
    assert plan["order_ready"] is False
    print(market, "매수 후 현금:", round(expected), "최소 현금:", round(ledger["reserve_krw"]))
    for leg in plan["sell_legs"] + plan["buy_legs"]:
        assert leg["quantity"] >= 1 and leg["quantity"] == int(leg["quantity"])
        print(leg["side"], leg["ticker"], leg["quantity"], leg["price_low"], leg["price_high"], leg["currency"])


kr 매수 후 현금: 1991388 최소 현금: 1878863
SELL 000660.KS 1 1210000 1216000 KRW
BUY 033780.KS 1 189000 190500 KRW
us 매수 후 현금: 5288551 최소 현금: 4952827
SELL TSM 6 424.01 426.13 USD
SELL MPWR 1 1268.82 1275.19 USD
SELL GEV 1 910.0 914.56 USD
BUY PG 1 154.19 154.29 USD
BUY ABBV 1 277.0 279.0 USD


## 과거 비교
현재 성장주 비중을 과거 경로에 적용해 수익률이 높게 나온다. 미래 예상수익이 아니다. KR 2024~2025는 GST 약 -50.8% 가격 단절로 제외하며 2026은 민감도에만 쓴다. 겹치는 기간은 독립 실험으로 세지 않는다.

In [3]:
comparison = historical.loc[historical["usable_as_selection_evidence"], ["market", "period", "policy", "cagr_pct", "max_drawdown_pct"]]
print(comparison.round(2).to_string(index=False))


market    period          policy  cagr_pct  max_drawdown_pct
    kr      2025            hold    192.92            -15.58
    kr      2025       sell_only    155.35            -12.67
    kr      2025 funded_rotation    156.16            -12.66
    us 2024_2025            hold     48.84            -26.95
    us 2024_2025       sell_only     30.03            -19.70
    us 2024_2025 funded_rotation     30.25            -19.71
    us      2025            hold     35.09            -25.15
    us      2025       sell_only     19.60            -17.53
    us      2025 funded_rotation     19.97            -17.55


## 세금 상한 민감도
기본 자산가치에서 세금 충당금은 비투자 현금으로 남는다. 상한 전액 납부를 가정하면 US 교체 손실이 유지보다 커질 수 있다.

In [4]:
print(stress.loc[stress["scenario"].eq("all_equity_minus20"), ["market", "policy", "pnl_krw", "pnl_if_upper_tax_paid_krw"]].round().to_string(index=False))


market          policy    pnl_krw  pnl_if_upper_tax_paid_krw
    kr            hold -1683532.0                 -1683532.0
    kr       sell_only -1450531.0                 -1450531.0
    kr funded_rotation -1489196.0                 -1489196.0
    us            hold -4736186.0                 -4736186.0
    us       sell_only -3522472.0                 -4906569.0
    us funded_rotation -3640218.0                 -5024314.0


## 판단
교체안은 현재 보유 유지보다 하락 노출과 상승 참여를 함께 줄였다.
매도 후 현금만 남기는 안에 비해 소액 신규 매수의 추가 개선은 작았다.
자금 조달 가능성이 매수의 투자 타당성을 증명하지 않는다. 논지·손익비·세금·결제를 확인한 뒤 판단한다.

## 실행 검증 기록
코드 셀을 순서대로 동일 Python 프로세스에서 실행하여 저장 출력과 단언문을 확인했다.
이 환경에 Jupyter 커널 패키지가 없어 nbclient 커널 실행 검증은 하지 않았다.
노트북을 여는 환경에서 Run All로 재검증할 수 있다.
